# Add Mutations to Merged Datasets

Adds mutation MB-selected features to each of the 8 existing Clinical+RNA+CNV merged files, then saves `4modalities_` copies.

**Run cells in order.**

**Input:** `MERGE/01_ultra_conservative.csv` ... `08_composite.csv`  
**Output:** same files updated with `MUT_` columns + `4modalities_` copies

## Setup — Imports, Paths, Best Mutations Config

In [3]:
"""
ADD MUTATIONS: Add mutation features to existing Clinical+RNA+CNV merged datasets.
Then save copies with prefix 4modalities_ for later comparison.

Script location: 01_Causal_feature_extraction/
Input/Output:    01_Causal_feature_extraction/MERGE/
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# PATHS
# ---------------------------------------------------------------------------
try:
    SCRIPT_DIR = Path(__file__).resolve().parent
except NameError:
    cwd = Path.cwd()
    SCRIPT_DIR = cwd.parent if cwd.name == "MERGE" else cwd

MUT_DIR   = SCRIPT_DIR / "Mutations"
FILT_DIR  = MUT_DIR / "statistical_filtered"
MB_DIR    = MUT_DIR / "mb_results"
MERGE_DIR = SCRIPT_DIR / "MERGE"

print(f"Script dir:  {SCRIPT_DIR}")
print(f"Mutations MB: {MB_DIR.exists()}")
print(f"MERGE dir:    {MERGE_DIR.exists()}")

# ---------------------------------------------------------------------------
# DATASET PAIRING: merged file -> Mutations dataset (matched by number 1-8)
# ---------------------------------------------------------------------------
DATASET_PAIRS = {
    "01_ultra_conservative": "mut_1_ultra_conservative_50genes",
    "02_conservative":       "mut_2_conservative_50genes",
    "03_standard":           "mut_3_standard_61genes",
    "04_fdr_significant":    "mut_4_fdr_significant_1000genes",
    "05_balanced":           "mut_5_balanced_50genes",
    "06_correlation":        "mut_6_correlation_50genes",
    "07_top_correlated":     "mut_7_top_correlated_200genes",
    "08_composite":          "mut_8_composite_1000genes",
}

# ---------------------------------------------------------------------------
# STEP 1: LOAD MUTATIONS SUMMARY AND SELECT BEST CONFIG PER DATASET
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 1: SELECT BEST MUTATIONS CONFIG PER DATASET (highest C-index)")
print("="*70)

summary = pd.read_csv(MB_DIR / "summary_all_results.csv")

best_mut = (
    summary
    .sort_values("c_index", ascending=False)
    .groupby("dataset", sort=False)
    .first()
    .reset_index()
)[["dataset", "algorithm", "alpha", "c_index", "n_causal_genes"]]

print(best_mut.to_string(index=False))

Script dir:  C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction
Mutations MB: True
MERGE dir:    True

STEP 1: SELECT BEST MUTATIONS CONFIG PER DATASET (highest C-index)
                         dataset algorithm  alpha  c_index  n_causal_genes
   mut_7_top_correlated_200genes      GSMB   0.20 0.563768              50
          mut_3_standard_61genes      IAMB   0.05 0.486162              50
      mut_2_conservative_50genes      GSMB   0.20 0.453401              50
mut_1_ultra_conservative_50genes      GSMB   0.05 0.417821              50
          mut_5_balanced_50genes      GSMB   0.10 0.405492              50
       mut_6_correlation_50genes      IAMB   0.10 0.393881              50
 mut_4_fdr_significant_1000genes      GSMB   0.05 0.390511              50
       mut_8_composite_1000genes      GSMB   0.05 0.390511              50


## Helper Functions

In [5]:
# ---------------------------------------------------------------------------
# STEP 2: HELPERS
# ---------------------------------------------------------------------------
def normalize_id(sample_id):
    """TCGA-D8-A146-01A  ->  TCGA-D8-A146"""
    parts = str(sample_id).split("-")
    return "-".join(parts[:3]) if len(parts) >= 3 else str(sample_id)


def load_genes(dataset, algorithm, alpha):
    for fmt in [f"{alpha:.2f}", str(alpha)]:
        p = MB_DIR / dataset / f"{algorithm}_alpha{fmt}_genes.txt"
        if p.exists():
            return [l.strip() for l in p.read_text().splitlines() if l.strip()]
    raise FileNotFoundError(
        f"Gene file not found: {MB_DIR / dataset / f'{algorithm}_alpha{alpha:.2f}_genes.txt'}"
    )


def load_mutations(mut_dataset, algorithm, alpha):
    """Load mutations data filtered to selected genes, normalized to patient IDs."""
    candidates = list(FILT_DIR.glob(f"{mut_dataset}*.csv"))
    if not candidates:
        raise FileNotFoundError(f"No mutations file found for '{mut_dataset}'")
    mut_file = candidates[0]

    genes     = load_genes(mut_dataset, algorithm, alpha)
    mut       = pd.read_csv(mut_file, index_col=0)
    available = [g for g in genes if g in mut.columns]
    missing_g = [g for g in genes if g not in mut.columns]
    if missing_g:
        print(f"  Warning: {len(missing_g)} mutation genes not in file: {missing_g}")
    if not available:
        raise ValueError(f"No genes available in {mut_file.name}")
    mut = mut[available].copy()
    print(f"  Mutations loaded: {mut.shape}  ({mut_file.name})")

    # Primary Tumor only
    tumor_mask = mut.index.str.contains(r"-01[A-Z]?$", regex=True)
    mut = mut[tumor_mask].copy()
    print(f"  After Primary Tumor filter: {len(mut)} samples")

    # normalize IDs
    mut.index = [normalize_id(i) for i in mut.index]

    # average duplicates
    if mut.index.duplicated().any():
        n_pts = mut.index[mut.index.duplicated(keep=False)].nunique()
        print(f"  Averaging {n_pts} patients with duplicate samples")
        mut = mut.groupby(mut.index).mean()

    # add MUT_ prefix
    mut.columns = [f"MUT_{c}" for c in mut.columns]
    return mut

print("\nHelper functions ready")


Helper functions ready


## Add Mutations to All 8 Datasets

In [7]:
# ---------------------------------------------------------------------------
# STEP 3: ADD MUTATIONS TO ALL 8 DATASETS
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 3: ADD MUTATIONS TO ALL 8 MERGED DATASETS")
print("="*70)

results = []

for short_name, mut_dataset in DATASET_PAIRS.items():
    merged_file = MERGE_DIR / f"{short_name}.csv"
    if not merged_file.exists():
        print(f"\nSkipping {short_name} — file not found")
        continue

    row = best_mut[best_mut["dataset"] == mut_dataset]
    if row.empty:
        print(f"\nNo mutations summary entry for {mut_dataset} — skipping")
        continue
    row = row.iloc[0]

    print(f"\n{'─'*70}")
    print(f"[{short_name}]")
    print(f"  Mutations dataset: {mut_dataset}")
    print(f"  Config: {row['algorithm']}  alpha={row['alpha']}  "
          f"C-index={row['c_index']:.4f}  genes={row['n_causal_genes']}")

    try:
        # load existing merged (Clinical + RNA + CNV)
        merged = pd.read_csv(merged_file, index_col=0)

        # drop any MUT_ columns from a previous run
        mut_cols_existing = [c for c in merged.columns if c.startswith("MUT_")]
        if mut_cols_existing:
            merged = merged.drop(columns=mut_cols_existing)
            print(f"  Dropped {len(mut_cols_existing)} existing MUT_ columns from previous run")

        n_clin = sum(1 for c in merged.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in merged.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in merged.columns if c.startswith("CNV_"))
        print(f"  Existing: {merged.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}")

        # load mutations
        mut = load_mutations(mut_dataset, row["algorithm"], float(row["alpha"]))

        # align and merge
        common = sorted(set(merged.index) & set(mut.index))
        print(f"  Common patients: {len(common)}  "
              f"(merged={len(merged)}, mutations={len(mut)})")
        if not common:
            raise ValueError("No common patients")

        final = pd.concat([merged.loc[common], mut.loc[common]], axis=1)

        # verify
        assert final.isna().sum().sum() == 0, \
            f"Missing values: {final.isna().sum().sum()}"
        n_mut  = sum(1 for c in final.columns if c.startswith("MUT_"))
        n_clin = sum(1 for c in final.columns if c.startswith("CLIN_"))
        n_rna  = sum(1 for c in final.columns if c.startswith("RNA_"))
        n_cnv  = sum(1 for c in final.columns if c.startswith("CNV_"))
        print(f"  Final: {final.shape}  "
              f"CLIN_={n_clin}  RNA_={n_rna}  CNV_={n_cnv}  MUT_={n_mut}  | no missing")

        # overwrite existing file
        final.to_csv(merged_file)
        print(f"  Saved: {merged_file.name}")

        results.append({
            "file":             short_name + ".csv",
            "n_samples":        final.shape[0],
            "n_clin_features":  n_clin,
            "n_rna_features":   n_rna,
            "n_cnv_features":   n_cnv,
            "n_mut_features":   n_mut,
            "n_total_features": final.shape[1],
            "mut_dataset":      mut_dataset,
            "mut_algorithm":    row["algorithm"],
            "mut_alpha":        row["alpha"],
            "mut_c_index":      row["c_index"],
        })

    except Exception as e:
        print(f"  ERROR: {e}")
        import traceback; traceback.print_exc()


STEP 3: ADD MUTATIONS TO ALL 8 MERGED DATASETS

──────────────────────────────────────────────────────────────────────
[01_ultra_conservative]
  Mutations dataset: mut_1_ultra_conservative_50genes
  Config: GSMB  alpha=0.05  C-index=0.4178  genes=50
  Dropped 54 existing MUT_ columns from previous run
  Existing: (923, 244)  CLIN_=144  RNA_=50  CNV_=50
  Mutations loaded: (948, 50)  (mut_1_ultra_conservative_50genes.csv)
  After Primary Tumor filter: 947 samples
  Common patients: 923  (merged=923, mutations=947)
  Final: (923, 294)  CLIN_=144  RNA_=50  CNV_=50  MUT_=50  | no missing
  Saved: 01_ultra_conservative.csv

──────────────────────────────────────────────────────────────────────
[02_conservative]
  Mutations dataset: mut_2_conservative_50genes
  Config: GSMB  alpha=0.2  C-index=0.4534  genes=50
  Dropped 57 existing MUT_ columns from previous run
  Existing: (923, 244)  CLIN_=144  RNA_=50  CNV_=50
  Mutations loaded: (948, 50)  (mut_2_conservative_50genes.csv)
  After Primar

## Save 4modalities_ Copies

In [9]:
# ---------------------------------------------------------------------------
# STEP 4: SAVE 4modalities_ COPIES
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 4: SAVE 4modalities_ COPIES")
print("="*70)
print("Saving copies with prefix 4modalities_ for later comparison\n")

for short_name in DATASET_PAIRS.keys():
    src = MERGE_DIR / f"{short_name}.csv"
    dst = MERGE_DIR / f"4modalities_{short_name}.csv"
    if src.exists():
        import shutil
        shutil.copy2(src, dst)
        print(f"  {src.name}  ->  {dst.name}")
    else:
        print(f"  Skipped (not found): {src.name}")

print(f"\nAll copies saved to: {MERGE_DIR}")


STEP 4: SAVE 4modalities_ COPIES
Saving copies with prefix 4modalities_ for later comparison

  01_ultra_conservative.csv  ->  4modalities_01_ultra_conservative.csv
  02_conservative.csv  ->  4modalities_02_conservative.csv
  03_standard.csv  ->  4modalities_03_standard.csv
  04_fdr_significant.csv  ->  4modalities_04_fdr_significant.csv
  05_balanced.csv  ->  4modalities_05_balanced.csv
  06_correlation.csv  ->  4modalities_06_correlation.csv
  07_top_correlated.csv  ->  4modalities_07_top_correlated.csv
  08_composite.csv  ->  4modalities_08_composite.csv

All copies saved to: C:\Users\olegk\Desktop\Thesis_v3\01_Causal_feature_extraction\MERGE


## Summary

In [11]:
# ---------------------------------------------------------------------------
# STEP 5: SUMMARY
# ---------------------------------------------------------------------------
print("\n" + "="*70)
print("STEP 5: SUMMARY")
print("="*70)

if results:
    df = pd.DataFrame(results)
    print(df.to_string(index=False))
    df.to_csv(MERGE_DIR / "merge_summary_4modalities.csv", index=False)

    print(f"\nAll 4 modalities per dataset:")
    print(f"  CLIN_ same:   {df['n_clin_features'].nunique() == 1}  "
          f"({df['n_clin_features'].iloc[0]} features)")
    print(f"  RNA_ varies:  {df['n_rna_features'].nunique() > 1}  "
          f"(range {df['n_rna_features'].min()}–{df['n_rna_features'].max()})")
    print(f"  CNV_ varies:  {df['n_cnv_features'].nunique() > 1}  "
          f"(range {df['n_cnv_features'].min()}–{df['n_cnv_features'].max()})")
    print(f"  MUT_ varies:  {df['n_mut_features'].nunique() > 1}  "
          f"(range {df['n_mut_features'].min()}–{df['n_mut_features'].max()})")
    print(f"  Total:        {df['n_total_features'].min()}–{df['n_total_features'].max()} features")
    print(f"  Samples:      {df['n_samples'].min()}–{df['n_samples'].max()}")
    print(f"\nFinal files in MERGE/:")
    print(f"  01_ultra_conservative.csv ... 08_composite.csv   (Clinical+RNA+CNV+MUT)")
    print(f"  4modalities_01_ultra_conservative.csv ... (same, renamed copies)")
    print(f"  merge_summary_4modalities.csv")
else:
    print("No datasets processed successfully.")


STEP 5: SUMMARY
                     file  n_samples  n_clin_features  n_rna_features  n_cnv_features  n_mut_features  n_total_features                      mut_dataset mut_algorithm  mut_alpha  mut_c_index
01_ultra_conservative.csv        923              144              50              50              50               294 mut_1_ultra_conservative_50genes          GSMB       0.05     0.417821
      02_conservative.csv        923              144              50              50              50               294       mut_2_conservative_50genes          GSMB       0.20     0.453401
          03_standard.csv        923              144              50              50              50               294           mut_3_standard_61genes          IAMB       0.05     0.486162
   04_fdr_significant.csv        923              144              50              50              50               294  mut_4_fdr_significant_1000genes          GSMB       0.05     0.390511
          05_balanced.csv   